

# Final Project : Apple Music Replay Analysis
## Jada Hilliard
*5.16.26*

### Academic Integrity Statements:
* I used past mini projects and Google CoLabs
* I used ChatGPT for assitance with code and adjusted as needed. This is clearly commented above each code.
* I used Google Gemini for assistance with code and adjusted as needed. This is clearly commented above each code.



In [1]:
# Import necessary packages here

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import datetime as datetime
import plotly.express as px

# Introduction & Uploading


For my project, I chose to analyze my own Apple Music Replay data. I decided on this data source because I knew it contained a large amount of information and personally interested me due to my love of music. The data was collected directly from Apple’s Data & Privacy portal and includes information such as song names, artists, timestamps, favorites, rankings, and play activity across many years of my listening history. I cleaned and transformed multiple datasets, merged information across files, and created visualizations to better understand patterns in my listening behavior. I wanted to explore how my music taste has changed over time, how favorited songs compare to non-favorited songs, and which songs quickly dominated my listening history after I first discovered them. In addition to techniques learned throughout the course, I also implemented a new technique called cumulative growth analysis in order to assess how quickly listening activity accumulated for certain songs relative to others over time.


My first step is to mount my google drive, and upload my Apple Music Replay CSV files by converting them into pandas dataframes.

In [2]:
#Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Convert CSV files into dataframes
playActivity = pd.read_csv("drive/My Drive/CS215-Spring-JADAHILLIARD/Projects/Final Project/Apple Music Play Activity.csv")
topContent = pd.read_csv("drive/My Drive/CS215-Spring-JADAHILLIARD/Projects/Final Project/Apple Music - Top Content.csv")
historyDaily = pd.read_csv("drive/My Drive/CS215-Spring-JADAHILLIARD/Projects/Final Project/Apple Music - Play History Daily Tracks.csv")
favorites = pd.read_csv("drive/My Drive/CS215-Spring-JADAHILLIARD/Projects/Final Project/Apple Music - Favorites.csv")

/tmp/ipykernel_2600/2238885193.py:2: DtypeWarning: Columns (6,18,19,52,73,88,97,98,99,101,102,103,112,116,117,122,123,135,136) have mixed types. Specify dtype option on import or set low_memory=False.
  playActivity = pd.read_csv("drive/My Drive/CS215-Spring-JADAHILLIARD/Projects/Final Project/Apple Music Play Activity.csv")
/tmp/ipykernel_2600/2238885193.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  historyDaily = pd.read_csv("drive/My Drive/CS215-Spring-JADAHILLIARD/Projects/Final Project/Apple Music - Play History Daily Tracks.csv")


# Examining & Cleaning

My next step is to examine each dataframe by displaying the first 5 rows. I do this using the pandas .head() function. Because some of my dataframes had an extensive amount of columns, I wanted to select only the ones I needed for this final project. Unfortunately, even trying to print all the columns didn't give me every column name. I used Google Gemini for code assistance to print all the column names as a list. Once I was able to see all the columns, I created a new dataframe from only the selected columns. I only had to do this for my playActivity and historyDaily dataframes.

In [4]:
#Examine first 5 rows
playActivity.head()

,Age Bucket,Album Name,Apple ID Number,Apple Music Subscription,Auto Play,Build Version,Bundle Version,Camera Option,Carrier Name,Client Build Version,...,Subscription User ID,Transition Type,Translation Displayed,Use Listening History,User's Audio Quality,User's Playback Format,User's Transition Type,UTC Offset In Seconds,Vocal Attenuation Duration,Vocal Attenuation Model ID
0,NaN,NaN,8050066180,True,AUTO_ON_CONTENT_UNSUPPORTED,"Music/3.1 iOS/16.2 model/iPhone14,2 hwp/t8110 ...",3.1,NaN,NaN,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,-28800,NaN,NaN
1,25-34,NaN,8050066180,True,AUTO_OFF,"Music/3.1 iOS/18.1.1 model/iPhone14,2 hwp/t811...",3.1,NaN,NaN,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,-28800,NaN,NaN
2,NaN,NaN,8050066180,True,AUTO_OFF,"Music/3.1 iOS/18.1.1 model/iPhone14,2 hwp/t811...",3.1,NaN,NaN,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,-28800,NaN,NaN
3,NaN,Memoirs of an Imperfect Angel,8050066180,True,NaN,"Music/3.1 iOS/11.2.2 model/iPhone10,1 hwp/t801...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,-28800,NaN,NaN
4,25-34,The Muppets (Original Motion Picture Soundtrack),8050066180,True,AUTO_OFF,"Music/3.1 iOS/17.3.1 model/iPhone14,2 hwp/t811...",3.1,NaN,NaN,NaN,...,1.608221e+09,NaN,NaN,False,NaN,NaN,NaN,-25200,158909.0,NaN


In [5]:
#Examine first 5 rows
topContent.head()

,Country,Content,Play Duration Milliseconds,First Played,Last Played,Last Received,Source Type,Rankings
0,United States,Radiohead,79764832,1740983106899,1771042225268,1771042225310,"IPHONE, MACINTOSH",1
1,United States,Deftones,62777095,1739940766078,1770507249469,1770500289236,"IPHONE, MACINTOSH",2
2,United States,Chris Stapleton,41531602,1739413526765,1770946797919,1770946798402,"IPHONE, MACINTOSH",3
3,United States,My Chemical Romance,38340795,1740528282218,1770361074989,1770361075032,"IPHONE, MACINTOSH",4
4,United States,Gracie Abrams,36365965,1739739053939,1770323296150,1770323136866,IPHONE,5


In [6]:
#Examine first 5 rows
historyDaily.head()

,Country,Track Identifier,Media type,Date Played,Hours,Play Duration Milliseconds,End Reason Type,Source Type,Play Count,Skip Count,Ignore For Recommendations,Track Reference,Track Description
0,United States,1025130621,NaN,20160101,7,267000,NATURAL_END_OF_TRACK,IPHONE,1,0,NaN,NaN,Drake - Hotline Bling
1,United States,1017804211,NaN,20160101,7,235000,NATURAL_END_OF_TRACK,IPHONE,1,0,NaN,NaN,The Weeknd - In the Night
2,United States,1049605634,NaN,20160101,7,233000,NATURAL_END_OF_TRACK,IPHONE,1,0,NaN,NaN,Justin Bieber - Love Yourself
3,United States,1040171617,NaN,20160101,8,230000,NATURAL_END_OF_TRACK,IPHONE,1,0,NaN,NaN,One Direction - Perfect
4,United States,1058944563,NaN,20160101,8,223000,NATURAL_END_OF_TRACK,IPHONE,1,0,NaN,NaN,LunchMoney Lewis - Ain't Too Cool


In [7]:
#Examine first 5 rows
favorites.head()

,Favorite Type,Item Reference,Item Description,Last Modified,Preference
0,Song,1097863219,Radiohead - Black Star,2026-02-13T22:14:31Z,NEUTRAL
1,Song,580695853,Matchbox Twenty - Push,2026-02-12T05:35:03Z,LIKE
2,Song,1474185294,TOOL - Prison Sex,2026-01-26T06:46:57Z,LIKE
3,Song,457503068,Kings of Leon - The End,2026-01-20T20:34:54Z,LIKE
4,Song,1576793811,Filter - Hey Man Nice Shot,2026-01-04T00:15:40Z,LIKE


In [8]:
#Code from Google Gemini, Lists every column name
list(playActivity.columns)

['Age Bucket',
 'Album Name',
 'Apple ID Number',
 'Apple Music Subscription',
 'Auto Play',
 'Build Version',
 'Bundle Version',
 'Camera Option',
 'Carrier Name',
 'Client Build Version',
 'Client Device Name',
 'Client IP Address',
 'Client Platform',
 'Container Album Name',
 'Container Artist Name',
 'Container Global Playlist ID',
 'Container ID',
 'Container iTunes Playlist ID',
 'Container Library ID',
 'Container Name',
 'Container Origin Type',
 'Container Personalized ID',
 'Container Playlist Folder ID',
 'Container Playlist ID',
 'Container Radio Station ID',
 'Container Radio Station Version',
 'Container Season ID',
 'Container Type',
 'Contingency',
 'Continuity Microphone Used',
 'Device App Name',
 'Device App Version',
 'Device Identifier',
 'Device OS Name',
 'Device OS Version',
 'Device Type',
 'Display Count',
 'Display Language',
 'Display Type',
 'End Position In Milliseconds',
 'End Reason Type',
 'Evaluation Variant',
 'Event End Timestamp',
 'Event ID',
 'Ev

In [9]:
#Create new dataframe with only selected columns
activity = playActivity[[ 'Song Name', 'Play Duration Milliseconds', 'IP City', 'IP Region Code', 'IP Latitude', 'IP Longitude', 'Event Start Timestamp', 'Event End Timestamp',  'End Reason Type' ]]

In [10]:
#Examine first 5 rows
activity.head()

,Song Name,Play Duration Milliseconds,IP City,IP Region Code,IP Latitude,IP Longitude,Event Start Timestamp,Event End Timestamp,End Reason Type
0,Drunk Last Night,NaN,NaN,NaN,NaN,NaN,2023-02-10T22:43:59.382Z,NaN,NaN
1,Traveller,NaN,WALLAWALLA,WA,46.1270,-118.3596,2025-02-11T18:47:22.176Z,NaN,NaN
2,Love on the Brain,0.0,REDMOND,WA,47.6833,-122.1231,2024-12-22T06:27:23.021Z,NaN,NaN
3,Obsessed,242230.0,NaN,NaN,NaN,NaN,2018-01-25T01:47:41.473Z,2018-01-25T01:51:43.703Z,NATURAL_END_OF_TRACK
4,Man Or Muppet,165443.0,REDMOND,WA,47.6833,-122.1231,2024-04-04T20:18:45.435Z,2024-04-04T20:21:30.878Z,OTHER


In [11]:
#Lists every column
list(historyDaily.columns)

['Country',
 'Track Identifier',
 'Media type',
 'Date Played',
 'Hours',
 'Play Duration Milliseconds',
 'End Reason Type',
 'Source Type',
 'Play Count',
 'Skip Count',
 'Ignore For Recommendations',
 'Track Reference',
 'Track Description']

In [12]:
#Create new dataframe from selected columns
history = historyDaily[['Date Played','Hours','Play Duration Milliseconds','End Reason Type','Play Count','Skip Count','Track Description']]

In [13]:
#Examine first 5 rows
history.head()

,Date Played,Hours,Play Duration Milliseconds,End Reason Type,Play Count,Skip Count,Track Description
0,20160101,7,267000,NATURAL_END_OF_TRACK,1,0,Drake - Hotline Bling
1,20160101,7,235000,NATURAL_END_OF_TRACK,1,0,The Weeknd - In the Night
2,20160101,7,233000,NATURAL_END_OF_TRACK,1,0,Justin Bieber - Love Yourself
3,20160101,8,230000,NATURAL_END_OF_TRACK,1,0,One Direction - Perfect
4,20160101,8,223000,NATURAL_END_OF_TRACK,1,0,LunchMoney Lewis - Ain't Too Cool


# Question 1 & Plotly Interactive

Now, I want to analyze how my listening habits have changed over time. I started by taking the play duration hours column, that was converted from milliseconds into hours, and grouping it by month. I then generated an interactive line graph using Plotly. The x-axis represents time in months, while the y-axis represents total play duration in hours. I added a title, axis labels, and other stylistic adjustments before displaying the graph. Finally, I exported the interactive visualization using .write_html() so it could be embedded into my GitHub Pages project website.

In [22]:
#Create new dataframe from grouping play duration hours by month
monthly_listening = history.groupby('Month')['Play Duration Hours'].sum().reset_index()

In [23]:
#Use Plotly to generate line plot
fig = px.line(
    monthly_listening,
    #Google Gemini assisted with debugging, changes datatype to string for plotting
    x=monthly_listening['Month'].astype(str),
    y='Play Duration Hours',
    #Add title
    title='Listening Hours Over Time',
    #Google Gemini assisted code, changes background to white
    template='plotly_white'
)

#Change x-axis and y-axis labels and change line color
fig.update_xaxes(title='Time')
fig.update_yaxes(title='Monthly Play Duration (Hours)')
#ChatGPT assisted code, adjusted as needed
fig.update_traces(line_color='hotpink')

#Display and download HTML for interactive graph
fig.show()
fig.write_html("monthly_listening_plot.html")

Narration of Visualization:

 The visualization shows my monthly listening activity across the last 10 years. While some periods show relatively steady listening behavior, other periods contain large spikes in total listening hours. I believe these spikes may be due to changes in routines, school workload, or periods where music played a larger role in my daily life. Overall, the line plot suggests that my listening behavior is not constant over time and instead changes substantially across different periods of my life.

# Question 2 & Plotly Interactive

I am interested in how the songs I mark as favorite compare to the songs I do not have marked as favorite. I call these favorited songs and non-favorited songs. I am interested in how features such as play duration, play count, and skip count differ between the two types of songs. In order to make these comparisons, I first needed to label the songs in my listening history as favorited or non-favorited. To do this, I used the .isin() function to create a new boolean column called “Is Favorite” that checked whether each song in the listening history dataframe also appeared in the favorites dataframe. Songs that appeared in both datasets were labeled as favorited, while all other songs were labeled as non-favorited. Before conducting the analysis, I cleaned and transformed several columns in the dataset. I removed rows with missing values, converted the “Date Played” column into datetime format, and created additional month and year columns from the date information. I also converted the “Play Duration Milliseconds” column into numeric format so it could be manipulated mathematically. In order to make the values easier to interpret, I created new columns converting play duration from milliseconds into both minutes and hours. Next, I created separate dataframes for favorited songs and non-favorited songs using booleans and I calculated the average play count, average skip count, and average play duration in minutes for each group of songs using the .mean() function. After calculating these averages, I created a new comparison dataframe to organize the results. I then used Plotly to generate an interactive grouped bar chart comparing favorited and non-favorited songs across the three features. I customized the graph by adjusting the title, colors, labels, and layout in order to make the visualization easier to interpret and more visually clear. I then downloaded the interactive graph as a HTML.

In [14]:
#Drop no anwser rows
history = history.dropna(subset=['Date Played', 'Track Description'])

In [15]:
#Create a new column to assign true if song is marked as favorite and false if not marked as favorite
history['Is Favorite'] = history['Track Description'].isin(favorites['Item Description'])

In [16]:
#Convert column to datetime format
history['Date Played'] = pd.to_datetime(history['Date Played'].astype(str), format='%Y%m%d')

In [17]:
#Create new columns for month and year using .to_period('M') and .year
history['Month'] = history['Date Played'].dt.to_period('M')
history['Year'] = history['Date Played'].dt.year

In [18]:
#Convert column to numeric in order to manipulate later on
history['Play Duration Milliseconds'] = pd.to_numeric(history['Play Duration Milliseconds'])

In [19]:
#Create new column of play duration in minutes from milliseconds column
history['Play Duration Minutes'] = history['Play Duration Milliseconds'] / 1000 / 60

In [20]:
#Create new column of play duration in hours from milliseconds column
history['Play Duration Hours'] = history['Play Duration Milliseconds'] / 1000 / 60 / 60

In [21]:
#Examine full dataframe
history

,Date Played,Hours,Play Duration Milliseconds,End Reason Type,Play Count,Skip Count,Track Description,Is Favorite,Month,Year,Play Duration Minutes,Play Duration Hours
0,2016-01-01,7,267000,NATURAL_END_OF_TRACK,1,0,Drake - Hotline Bling,True,2016-01,2016,4.450000,0.074167
1,2016-01-01,7,235000,NATURAL_END_OF_TRACK,1,0,The Weeknd - In the Night,False,2016-01,2016,3.916667,0.065278
2,2016-01-01,7,233000,NATURAL_END_OF_TRACK,1,0,Justin Bieber - Love Yourself,False,2016-01,2016,3.883333,0.064722
3,2016-01-01,8,230000,NATURAL_END_OF_TRACK,1,0,One Direction - Perfect,False,2016-01,2016,3.833333,0.063889
4,2016-01-01,8,223000,NATURAL_END_OF_TRACK,1,0,LunchMoney Lewis - Ain't Too Cool,False,2016-01,2016,3.716667,0.061944
...,...,...,...,...,...,...,...,...,...,...,...,...
83449,2026-02-12,21,0,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,0,1,Mitski - I Bet on Losing Dogs,True,2026-02,2026,0.000000,0.000000
83450,2026-02-12,16,0,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,0,1,Radiohead - Fake Plastic Trees,True,2026-02,2026,0.000000,0.000000
83451,2026-02-12,16,0,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,0,1,Pierce the Veil - Emergency Contact,False,2026-02,2026,0.000000,0.000000
83452,2026-02-12,16,0,TRACK_SKIPPED_FORWARDS,0,1,Gracie Abrams - That’s So True,True,2026-02,2026,0.000000,0.000000


In [24]:
#Examine full dataframe
favorites

,Favorite Type,Item Reference,Item Description,Last Modified,Preference
0,Song,1097863219,Radiohead - Black Star,2026-02-13T22:14:31Z,NEUTRAL
1,Song,580695853,Matchbox Twenty - Push,2026-02-12T05:35:03Z,LIKE
2,Song,1474185294,TOOL - Prison Sex,2026-01-26T06:46:57Z,LIKE
3,Song,457503068,Kings of Leon - The End,2026-01-20T20:34:54Z,LIKE
4,Song,1576793811,Filter - Hey Man Nice Shot,2026-01-04T00:15:40Z,LIKE
...,...,...,...,...,...
102,Song,610183913,The Paper Kites - Bloom (Bonus Track),2015-11-05T05:13:25.770Z,LIKE
103,Album,1081578077,Shawn Mendes - Handwritten (Deluxe),2016-02-04T00:25:03Z,LIKE
104,Album,1078293237,ZAYN - PILLOWTALK - Single,2016-02-04T00:21:22Z,LIKE
105,Playlist,pl.7e6597c750f342938c14c97279e84d42,NaN,2016-03-01T04:25:46.493Z,LIKE


In [25]:
#Create new dataframe from songs that are marked as favorite
favorited = history[history['Is Favorite'] == True]
favorited

,Date Played,Hours,Play Duration Milliseconds,End Reason Type,Play Count,Skip Count,Track Description,Is Favorite,Month,Year,Play Duration Minutes,Play Duration Hours
0,2016-01-01,7,267000,NATURAL_END_OF_TRACK,1,0,Drake - Hotline Bling,True,2016-01,2016,4.450000,0.074167
9,2016-01-01,7,192000,NATURAL_END_OF_TRACK,1,0,Daya - Hide Away,True,2016-01,2016,3.200000,0.053333
11,2016-01-01,7,35000,TRACK_SKIPPED_FORWARDS,1,1,Shawn Mendes - Stitches,True,2016-01,2016,0.583333,0.009722
17,2016-01-02,8,267000,NATURAL_END_OF_TRACK,1,0,Drake - Hotline Bling,True,2016-01,2016,4.450000,0.074167
18,2016-01-02,8,263000,NATURAL_END_OF_TRACK,1,0,James Bay - Let It Go,True,2016-01,2016,4.383333,0.073056
...,...,...,...,...,...,...,...,...,...,...,...,...
83445,2026-02-12,5,79335,PLAYBACK_MANUALLY_PAUSED,1,0,Matchbox Twenty - Push,True,2026-02,2026,1.322250,0.022037
83448,2026-02-12,16,0,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,0,1,Sam Phillips - Reflecting Light,True,2026-02,2026,0.000000,0.000000
83449,2026-02-12,21,0,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,0,1,Mitski - I Bet on Losing Dogs,True,2026-02,2026,0.000000,0.000000
83450,2026-02-12,16,0,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,0,1,Radiohead - Fake Plastic Trees,True,2026-02,2026,0.000000,0.000000


In [26]:
#Find mean play count for favorited songs
fav_plays_mean = favorited['Play Count'].mean()
fav_plays_mean

np.float64(1.5140891945861992)

In [27]:
#Find mean skip count for favorited songs
fav_skips_mean = favorited['Skip Count'].mean()
fav_skips_mean

np.float64(0.5174173507876636)

In [28]:
#Find mean play duration in minutes for favorited songs
fav_mins_mean = favorited['Play Duration Minutes'].mean()
fav_mins_mean

np.float64(5.6091545484801415)

In [29]:
#Create new dataframe showing means of favorited songs
comparison = pd.DataFrame({'Favorites': [fav_plays_mean, fav_skips_mean, fav_mins_mean]})

In [30]:
#Set means as index
comparison.index = ['Average Play Count', 'Average Skip Count', 'Average Play Duration Minutes']
comparison

,Favorites
Average Play Count,1.514089
Average Skip Count,0.517417
Average Play Duration Minutes,5.609155


In [31]:
#Create new dataframe from songs that are not marked as favorite
non_favorited = history[history['Is Favorite'] == False]
non_favorited

,Date Played,Hours,Play Duration Milliseconds,End Reason Type,Play Count,Skip Count,Track Description,Is Favorite,Month,Year,Play Duration Minutes,Play Duration Hours
1,2016-01-01,7,235000,NATURAL_END_OF_TRACK,1,0,The Weeknd - In the Night,False,2016-01,2016,3.916667,0.065278
2,2016-01-01,7,233000,NATURAL_END_OF_TRACK,1,0,Justin Bieber - Love Yourself,False,2016-01,2016,3.883333,0.064722
3,2016-01-01,8,230000,NATURAL_END_OF_TRACK,1,0,One Direction - Perfect,False,2016-01,2016,3.833333,0.063889
4,2016-01-01,8,223000,NATURAL_END_OF_TRACK,1,0,LunchMoney Lewis - Ain't Too Cool,False,2016-01,2016,3.716667,0.061944
5,2016-01-01,7,215000,NATURAL_END_OF_TRACK,1,0,JoJo - When Love Hurts,False,2016-01,2016,3.583333,0.059722
...,...,...,...,...,...,...,...,...,...,...,...,...
83444,2026-02-12,23,169336,NATURAL_END_OF_TRACK,1,0,Jake Owen - Barefoot Blue Jean Night,False,2026-02,2026,2.822267,0.047038
83446,2026-02-12,5,55434,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,1,1,Matchbox Twenty - 3 am,False,2026-02,2026,0.923900,0.015398
83447,2026-02-12,20,0,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,0,1,Miley Cyrus - Used To Be Young,False,2026-02,2026,0.000000,0.000000
83451,2026-02-12,16,0,MANUALLY_SELECTED_PLAYBACK_OF_A_DIFF_ITEM,0,1,Pierce the Veil - Emergency Contact,False,2026-02,2026,0.000000,0.000000


In [32]:
#Find mean play count for favorited songs
non_plays_mean = non_favorited['Play Count'].mean()
non_plays_mean

np.float64(1.3865458921897764)

In [33]:
#Find mean skip count for favorited songs
non_skips_mean = non_favorited['Skip Count'].mean()
non_skips_mean

np.float64(0.5326914549946158)

In [34]:
#Find mean play duartion in minutes for favorited songs
non_mins_mean = non_favorited['Play Duration Minutes'].mean()
non_mins_mean

np.float64(4.592087624627859)

In [35]:
#Create new column showing means of non-favorited songs
comparison['Non-Favorites'] = [non_plays_mean, non_skips_mean, non_mins_mean]
comparison

,Favorites,Non-Favorites
Average Play Count,1.514089,1.386546
Average Skip Count,0.517417,0.532691
Average Play Duration Minutes,5.609155,4.592088


In [36]:
#Use Plotly to visualize bar graph favorited songs and non-favorited songs
fig = px.bar(comparison,
             barmode = 'group',
             template = 'plotly_white',
             title = 'Features of Favorited vs Non-Favorited Songs in Listening History',
             #ChatGPT assisted code and adjusted as needed, changes colors of bars
             color_discrete_sequence = ['pink', 'lightblue'])

#ChatGPT assisted code and adjusted as needed, resizes layout
fig.update_layout(
    width=900,
    height=500
)

#Adds labels for x-axis and y-axis
fig.update_xaxes(title='Feature Type')
fig.update_yaxes(title='Average Value')

#Display interactive graph and download HTML
fig.show()
fig.write_html("favorite_vs_nonfavorite_plot.html")

Narration of Visualiztion:

According to the interactive bar graph, comparing all the songs I've listened to, songs I marked as favorite have, on average, higher play counts and higher play duration (in minutes) than songs not marked as favorite. Favorited songs also have a slightly smaller average skip count. This suggests that favorited songs are played more often and listened to for a fuller amount of time. I found it interesting that the average skip count was pretty similar which suggests I skip my favorited songs about the same time as I skip non-favorited songs. The interactive bar graph is designed so one can hover over to see exact values.


# Question 3 & New technique

Now I want to investigate how songs rapidly become dominant in my listening history. I used a cumulative growth analysis approach and visualized the results using a bubble chart. I learned about cumulative growth analysis and visualizing with a Plotly bubble chart from ChatGPT when asking about new techniques or visualizations. ChatGPT helped me understand what it means and why it was benefical for analyzing my data the way I wanted. It also helped me understand how a bubble chart works and how it could help visualize my data. Rather than only examining total listening hours, cumulative growth analysis allowed me to analyze how quickly songs accumulated listening activity relative to time after discovery, or after the first time listening to it. To begin this analysis, I grouped the listening history data by song title and identified the first date each song appeared in my listening history. I converted the date columns into datetime format in order to calculate the number of days since discovery for every song. I created a new column called “Hours Per Day” by dividing total listening hours by the number of days since discovery. This measurement served as a listening growth rate that allowed me to compare how quickly songs became dominant parts of my listening history over time. I decided to only choose songs that I had listened to longer than 30 days because I noticed songs that I had only listened to for 1-2 days had a very large growth rate. I assumed this is due to being recently discovered. Finally, I used Plotly to create an interactive bubble chart visualization. In the graph, the x-axis represents days since first time listening to the songs, the y-axis represents total listening hours, and the size and color of each bubble represent listening growth rate measured in hours per day. Larger and brighter bubbles therefore indicate songs that rapidly accumulated listening activity shortly after discovery. I downloaded the interactive graph as an HTML.

In [37]:
#Create new dataframe by grouping song name by first date ever played
song_first = history.groupby(
    'Track Description'
)['Date Played'].min().reset_index()
song_first

,Track Description,Date Played
0,"""Weird Al"" Yankovic - Fat",2020-06-16
1,$NOT & A$AP Rocky - Doja,2022-03-15
2,$uicideboy$ & Germ - Slenderman,2020-06-30
3,"$uicideboy$ - ...And to Those I Love, Thanks f...",2020-09-04
4,'Til Tuesday - Voices Carry (Single Mix),2018-01-25
...,...,...
9716,Óscar Maydon & Fuerza Regida - Tu Boda,2025-04-18
9717,עפרה חזה & Eden Riegel - Deliver Us,2023-02-14
9718,張信哲 - 愛你的宿命,2020-07-13
9719,아가동요 - Cute Puppy,2020-07-13


In [38]:
#Creating copy of column and deleting old in order to rename
song_first['First Played'] = song_first['Date Played']
song_first = song_first.drop(columns=['Date Played'])
song_first

,Track Description,First Played
0,"""Weird Al"" Yankovic - Fat",2020-06-16
1,$NOT & A$AP Rocky - Doja,2022-03-15
2,$uicideboy$ & Germ - Slenderman,2020-06-30
3,"$uicideboy$ - ...And to Those I Love, Thanks f...",2020-09-04
4,'Til Tuesday - Voices Carry (Single Mix),2018-01-25
...,...,...
9716,Óscar Maydon & Fuerza Regida - Tu Boda,2025-04-18
9717,עפרה חזה & Eden Riegel - Deliver Us,2023-02-14
9718,張信哲 - 愛你的宿命,2020-07-13
9719,아가동요 - Cute Puppy,2020-07-13


In [39]:
#Create new dataframe by grouping track description by sum of play duration hours
song_hours = history.groupby(
    'Track Description'
)['Play Duration Hours'].sum()

song_hours

,Play Duration Hours
Track Description,
"""Weird Al"" Yankovic - Fat",0.149425
$NOT & A$AP Rocky - Doja,0.059472
$uicideboy$ & Germ - Slenderman,0.064722
"$uicideboy$ - ...And to Those I Love, Thanks for Sticking Around",0.124584
'Til Tuesday - Voices Carry (Single Mix),0.517377
...,...
Óscar Maydon & Fuerza Regida - Tu Boda,0.032434
עפרה חזה & Eden Riegel - Deliver Us,0.026746
張信哲 - 愛你的宿命,0.008611


In [40]:
#Merge dataframes using track description as main column
songs = song_first.merge(
    song_hours,
    on='Track Description'
)

songs

,Track Description,First Played,Play Duration Hours
0,"""Weird Al"" Yankovic - Fat",2020-06-16,0.149425
1,$NOT & A$AP Rocky - Doja,2022-03-15,0.059472
2,$uicideboy$ & Germ - Slenderman,2020-06-30,0.064722
3,"$uicideboy$ - ...And to Those I Love, Thanks f...",2020-09-04,0.124584
4,'Til Tuesday - Voices Carry (Single Mix),2018-01-25,0.517377
...,...,...,...
9716,Óscar Maydon & Fuerza Regida - Tu Boda,2025-04-18,0.032434
9717,עפרה חזה & Eden Riegel - Deliver Us,2023-02-14,0.026746
9718,張信哲 - 愛你的宿命,2020-07-13,0.008611
9719,아가동요 - Cute Puppy,2020-07-13,0.016111


In [41]:
#Set today as maximun date played
today = history['Date Played'].max()
#Create new column of days since first discovery of song
songs['Days Since Discovery'] = ( today - songs['First Played']).dt.days
#Display dataframe
songs

,Track Description,First Played,Play Duration Hours,Days Since Discovery
0,"""Weird Al"" Yankovic - Fat",2020-06-16,0.149425,2067
1,$NOT & A$AP Rocky - Doja,2022-03-15,0.059472,1430
2,$uicideboy$ & Germ - Slenderman,2020-06-30,0.064722,2053
3,"$uicideboy$ - ...And to Those I Love, Thanks f...",2020-09-04,0.124584,1987
4,'Til Tuesday - Voices Carry (Single Mix),2018-01-25,0.517377,2940
...,...,...,...,...
9716,Óscar Maydon & Fuerza Regida - Tu Boda,2025-04-18,0.032434,300
9717,עפרה חזה & Eden Riegel - Deliver Us,2023-02-14,0.026746,1094
9718,張信哲 - 愛你的宿命,2020-07-13,0.008611,2040
9719,아가동요 - Cute Puppy,2020-07-13,0.016111,2040


In [42]:
#Sort values by column, ascending
songs = songs.sort_values(by='Days Since Discovery')
songs

,Track Description,First Played,Play Duration Hours,Days Since Discovery
44,3 Doors Down - When I'm Gone,2026-02-11,0.021683,1
41,3 Doors Down - Here Without You,2026-02-11,1.428966,1
43,3 Doors Down - Kryptonite,2026-02-11,0.000000,1
42,3 Doors Down - It's Not My Time,2026-02-11,0.000000,1
764,"Bad Bunny, Jowell & Randy & Ñengo Flow - Safaera",2026-02-10,0.038909,2
...,...,...,...,...
8785,The Weeknd - In the Night,2016-01-01,5.171806,3695
2147,Daya - Hide Away,2016-01-01,0.713457,3695
8988,Tove Lo - Talking Body,2016-01-01,2.219341,3695
6300,One Direction - Perfect,2016-01-01,0.447148,3695


In [43]:
#Create new column of listening growth by taking play duration hours divided by days since discovery
songs['Hours Per Day'] = ( songs['Play Duration Hours'] / songs['Days Since Discovery'])
songs

,Track Description,First Played,Play Duration Hours,Days Since Discovery,Hours Per Day
44,3 Doors Down - When I'm Gone,2026-02-11,0.021683,1,0.021683
41,3 Doors Down - Here Without You,2026-02-11,1.428966,1,1.428966
43,3 Doors Down - Kryptonite,2026-02-11,0.000000,1,0.000000
42,3 Doors Down - It's Not My Time,2026-02-11,0.000000,1,0.000000
764,"Bad Bunny, Jowell & Randy & Ñengo Flow - Safaera",2026-02-10,0.038909,2,0.019455
...,...,...,...,...,...
8785,The Weeknd - In the Night,2016-01-01,5.171806,3695,0.001400
2147,Daya - Hide Away,2016-01-01,0.713457,3695,0.000193
8988,Tove Lo - Talking Body,2016-01-01,2.219341,3695,0.000601
6300,One Direction - Perfect,2016-01-01,0.447148,3695,0.000121


In [48]:
#Create new dataframe of sorted growth rate
top = songs.sort_values(by='Hours Per Day')

top

,Track Description,First Played,Play Duration Hours,Days Since Discovery,Hours Per Day
6230,Of Mice & Men - Timeless,2022-11-02,0.000000,1198,0.000000
3730,Jameson Rodgers - Midnight Daydream,2016-08-04,0.000000,3479,0.000000
9287,Walter Martin - Costa Rica,2016-08-04,0.000000,3479,0.000000
8888,Timeflies - Once In A While,2016-08-04,0.000000,3479,0.000000
1181,Brett Eldredge - Wanna Be That Song,2016-08-04,0.000000,3479,0.000000
...,...,...,...,...,...
6025,"Nat ""King"" Cole - If I May",2025-12-09,1.312613,65,0.020194
1164,Breaking Benjamin - Dear Agony,2024-08-09,12.031015,552,0.021795
8705,The Smashing Pumpkins - Mayonaise,2025-09-09,3.624710,156,0.023235
4986,Loathe - A Sad Cartoon,2022-06-17,36.667042,1336,0.027445


In [45]:
#Select only songs that have been listened to longer than 30 days
songs = songs[
    songs['Days Since Discovery'] >= 30]

In [46]:
#Create new dataframe of sorted growth rate
top = songs.sort_values( by='Hours Per Day', ascending=False)

top

,Track Description,First Played,Play Duration Hours,Days Since Discovery,Hours Per Day
6904,Radiohead - Black Star,2024-12-04,14.653865,435,0.033687
4986,Loathe - A Sad Cartoon,2022-06-17,36.667042,1336,0.027445
8705,The Smashing Pumpkins - Mayonaise,2025-09-09,3.624710,156,0.023235
1164,Breaking Benjamin - Dear Agony,2024-08-09,12.031015,552,0.021795
6025,"Nat ""King"" Cole - If I May",2025-12-09,1.312613,65,0.020194
...,...,...,...,...,...
482,Aqua - Barbie Girl (Radio),2017-06-18,0.000000,3161,0.000000
4350,Katy Perry - Witness,2017-06-18,0.000000,3161,0.000000
1087,Bo Bice - Blades of Glory,2017-06-29,0.000000,3150,0.000000
8836,Theodore Shapiro - Snow Cones,2017-06-29,0.000000,3150,0.000000


In [47]:
#Use Plotly to generate a bubble scatterplot
fig = px.scatter(
    top,
    x='Days Since Discovery',
    y='Play Duration Hours',
    size='Hours Per Day',
    color='Hours Per Day',
    hover_name='Track Description',
    template='plotly_white',
    title='Listening Growth Patterns After First Listening to Song')

#Label for x-axis and y-axis
fig.update_xaxes(title='Days Since First Listen')
fig.update_yaxes(title='Total Listening Hours')

#Display interactive graph and download HTML
fig.show()
fig.write_html("growth_plot.html")

Narration of Visualization:

 The bubble chart displays each song in my listening history and how many total hours I spent listening to each song. The songs with the largest listening growths are represented by the larger yellow bubbles. The number of days since I first discovered each song are on the x-axis, while total listening hours are on the y-axis. Most songs are near the bottom of the graph, indicating relatively low long-term listening accumulation. However, several songs stand out with both large listening totals and high listening growth rates, suggesting that they rapidly became major parts of my listening habits shortly after discovery. This visualization highlights how some songs gradually accumulate listening activity while others quickly dominate listening behavior.

 One interesting pattern I noticed is that many of the smallest bubbles appear toward both the bottom and the left side of the graph, while many of the larger bubbles are associated with more recently discovered songs. I think this may suggest that over time I have developed a stronger understanding of my music taste and have become better at finding songs that I strongly connect with. As a result, more recently discovered songs may accumulate listening hours and listening growth more rapidly than songs I discovered in earlier years.


Reflection:

I had a lot of fun working on this final project. I enjoyed combined my interest in music with data analysis and visualization techniques. I was surprised by how much information was included within the Apple Music Replay datasets. It took me a while to go through and make sense of them, and organize by which ones I felt were appropitate for this project. One of the most surprising findings was discovering that I listened to 173 hours of music in one month.